# NYC Taxi | PySpark exploration

## Goal
Practice PySpark on `samples.nyctaxi.trips`: inspect the schema, calculate duration, identify quality issues, and group records by pickup ZIP code.

## Setup
This is an **optional introductory notebook**, independent of the January 2025 pipeline. The recorded sample contains 21,932 records from 2016. Its results and eligibility rules must not be combined with the January 2025 results in notebooks 02–05.

Run in Databricks with Python as the default notebook language. Selected table outputs below are recorded outputs from the original export, not a fresh execution of this packaged notebook.

## Steps

### Load the Databricks sample table

In [ ]:
taxi = spark.read.table("samples.nyctaxi.trips")
display(taxi.limit(10))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222


### Inspect the schema

In [ ]:
taxi.printSchema()

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



### Calculate duration in minutes

In [ ]:
from pyspark.sql.functions import  col
 
taxi_sure = taxi.withColumn (
    "sure_dakika",
    (col("tpep_dropoff_datetime").cast("long")
     - col("tpep_pickup_datetime").cast("long")) / 60
) 
display(taxi_sure.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "sure_dakika"
).limit(10))

tpep_pickup_datetime,tpep_dropoff_datetime,sure_dakika
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,9.366666666666667
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,8.233333333333333
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,11.566666666666666
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,14.183333333333334
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,30.616666666666667
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,5.15
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,23.0
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,16.633333333333333
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,19.266666666666666
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,13.366666666666667


### Inspect suspicious durations

In [ ]:
display(
    taxi_sure
    .filter(
        col("sure_dakika").isNull()
        | (col("sure_dakika") <= 0)
        | (col("sure_dakika") > 180)
    )
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "sure_dakika"
    )
    .limit(10)
)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,sure_dakika
2016-02-18T19:05:22.000Z,2016-02-19T18:40:19.000Z,2.05,1414.95
2016-02-18T03:48:28.000Z,2016-02-19T03:29:35.000Z,12.99,1421.1166666666666
2016-01-14T01:42:42.000Z,2016-01-15T01:18:09.000Z,1.75,1415.45
2016-01-02T02:38:34.000Z,2016-01-03T02:22:50.000Z,3.29,1424.2666666666667
2016-01-10T04:50:12.000Z,2016-01-11T04:37:07.000Z,11.11,1426.9166666666667
2016-01-02T22:15:22.000Z,2016-01-03T21:52:36.000Z,17.3,1417.2333333333333
2016-02-26T18:55:23.000Z,2016-02-27T18:50:30.000Z,0.74,1435.1166666666666
2016-02-27T19:52:01.000Z,2016-02-28T00:00:00.000Z,1.47,247.98333333333332
2016-02-18T12:50:59.000Z,2016-02-19T12:25:33.000Z,1.0,1414.5666666666666
2016-01-30T23:09:45.000Z,2016-01-31T23:08:36.000Z,4.1,1438.85


### Summarize duration quality

In [ ]:
from pyspark.sql import functions as F

display(
    taxi_sure.agg(
        F.count("*").alias("toplam_kayit"),
        F.sum(F.when(F.col("sure_dakika").isNull(), 1).otherwise(0)).alias("bos_sure"),
        F.sum(F.when(F.col("sure_dakika") <= 0, 1).otherwise(0)).alias("sifir_veya_negatif"),
        F.sum(F.when(
            (F.col("sure_dakika") > 180) & (F.col("sure_dakika") <= 1200), 1
        ).otherwise(0)).alias("uc_ile_yirmi_saat_arasi"),
        F.sum(F.when(F.col("sure_dakika") > 1200, 1).otherwise(0)).alias("yirmi_saatten_uzun")
    )
)

toplam_kayit,bos_sure,sifir_veya_negatif,uc_ile_yirmi_saat_arasi,yirmi_saatten_uzun
21932,0,1,4,29


### Inspect non-positive and long durations

In [ ]:
display(
    taxi_sure
    .filter(
        (F.col("sure_dakika") <= 0)
        | (
            (F.col("sure_dakika") > 180)
            & (F.col("sure_dakika") <= 1200)
        )
    )
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "sure_dakika"
    )
    .orderBy("sure_dakika")
    .limit(20)
)

### Label duration suitability

In [ ]:
taxi_kalite = taxi_sure.withColumn(
    "sure_durumu",
    F.when(F.col("sure_dakika").isNull(), "bos")
     .when(F.col("sure_dakika") <= 0, "sifir_veya_negatif")
     .when(F.col("sure_dakika") > 180, "uc_saatten_uzun")
     .otherwise("uygun")
)

display(
    taxi_kalite
    .groupBy("sure_durumu")
    .count()
    .orderBy("sure_durumu")
)

sure_durumu,count
sifir_veya_negatif,1
uc_saatten_uzun,33
uygun,21898


### Check distance and fare among duration candidates

In [ ]:
taxi_sure_uygun = taxi_kalite.filter(F.col("sure_durumu") == "uygun")

display(
    taxi_sure_uygun.agg(
        F.count("*").alias("sure_acisindan_uygun"),
        F.sum(F.when(
            F.col("trip_distance").isNull() | (F.col("trip_distance") <= 0),
            1
        ).otherwise(0)).alias("bos_veya_sifir_mesafe"),
        F.sum(F.when(
            F.col("fare_amount").isNull() | (F.col("fare_amount") < 0),
            1
        ).otherwise(0)).alias("bos_veya_negatif_ucret")
    )
)

sure_acisindan_uygun,bos_veya_sifir_mesafe,bos_veya_negatif_ucret
21898,75,5


### Compare overlapping issues

In [ ]:
taxi_kontrol = (
    taxi_sure_uygun
    .withColumn(
        "mesafe_sorunu",
        F.col("trip_distance").isNull() | (F.col("trip_distance") <= 0)
    )
    .withColumn(
        "ucret_sorunu",
        F.col("fare_amount").isNull() | (F.col("fare_amount") < 0)
    )
)

display(
    taxi_kontrol
    .groupBy("mesafe_sorunu", "ucret_sorunu")
    .count()
    .orderBy("mesafe_sorunu", "ucret_sorunu")
)

mesafe_sorunu,ucret_sorunu,count
false,false,21818
false,true,5
true,false,75


### Summarize the introductory analysis subset

In [ ]:
taxi_analiz = taxi_kontrol.filter(
    (~F.col("mesafe_sorunu")) & (~F.col("ucret_sorunu"))
)

display(
    taxi_analiz.agg(
        F.count("*").alias("analize_uygun_kayit"),
        F.round(F.avg("sure_dakika"), 2).alias("ortalama_sure_dakika"),
        F.round(F.avg("trip_distance"), 2).alias("ortalama_mesafe"),
        F.round(F.avg("fare_amount"), 2).alias("ortalama_ucret")
    )
)

analize_uygun_kayit,ortalama_sure_dakika,ortalama_mesafe,ortalama_ucret
21818,13.18,2.86,12.31


### Compare pickup ZIP codes

In [ ]:
display(
    taxi_analiz
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("yolculuk_sayisi"),
        F.round(F.avg("sure_dakika"), 2).alias("ortalama_sure"),
        F.round(F.avg("fare_amount"), 2).alias("ortalama_ucret")
    )
    .orderBy(F.desc("yolculuk_sayisi"))
    .limit(10)
)

pickup_zip,yolculuk_sayisi,ortalama_sure,ortalama_ucret
10001,1225,12.2,10.62
10003,1180,12.45,10.99
10011,1126,12.49,10.87
10021,1017,11.53,10.15
10018,1008,12.59,11.42
10023,1005,10.97,10.06
10028,926,10.78,10.17
10012,831,13.25,11.38
10110,761,12.25,10.86
10065,697,10.9,9.77


## Checks and takeaways
The recorded sample analysis selected 21,818 records after duration, distance, and fare checks. This is a teaching subset of the sample data, **not** the January 2025 measurement population.

## Next steps
Use notebook 02 to start the independent pipeline from the TLC January 2025 Parquet and taxi-zone CSV files. Notebook 01 creates no persistent project table.